# Frequency Shift Keying Project - Part 4: Complete System

In [ ]:
## loa numpy and matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from IPython.display import Audio

## Background

A communications system aims to send information from one location to
another. In a digital communications system, the information consists
of a sequence of bits. Simply put, a digital communications system
uses the following steps to accomplish that goal:
* The transmitter is provided a sequence of bits that must be sent
  to the receiver; the receiver is in a different location from the
  transmitter.
* The transmitter maps the sequence of bits into an analog signal
  or waveform;
* The signal is sent to the receiver; this may happen via a cabled
  connection or wirelessly. As the signal travels from transmitter to
  the receiver it becomes weaker and it is disturbed by *noise*.
* The receiver observes this signal and employs knowledge of how
  the transmitter maps bits to signals to reconstruct the original bit
  sequence. 

In Parts 1 and 2, you explored how textual information is mapped to a
signal that is suitable for transmission over either a cable or
wirelessly. In Part 3, you learned about matched filters that make up the frontend
of the receiver. 

In lab, Part 4, you will first complete the receiver by
adding a detector that considers the matched filter outputs and
decides wether a 0-bit or a 1-bit was transmitted.
You will then combine the transmitter and the complete receiver
(together with a provided channel simulator) to simulate a complete
FSK system.


### Import Code from Parts 1 through 3

This part of the project builds on the functions that you developed in parts 1 through 3. 

To make these functions accessible, we need to store them in a *module*. To do so, proceed as follows.

1. Copy the file `fsk.py` that you created for part 3 into the directory that contains this notebook.
  + no changes are required
  + this file should already contain
    - an instruction to import Numpy
    - function definitions for functions `int_to_bits()`, `bits_to_int()`, `string_to_bits()`, `bits_to_string()`, `fsk_modulator()`, and `fsk_transmitter()`

  To import your code and to verify that the imports worked, execute the following cell: 


In [ ]:
# import the top-level functions
from fsk import string_to_bits, bits_to_string, fsk_transmitter, fsk_modulator

# test bit to/from string conversions
str = "Information"
bits = string_to_bits(str)
assert(str == bits_to_string(bits))

# test FSK signal generation
f0 = 1e2
f1 = 2e2
T = 20e-3
fs = 10e3
sig = fsk_transmitter(str, f0, f1, T, fs)

assert(len(sig) == len(bits) * int(T * fs))

print("OK")

## Complete Receiver

In the previous lab, we looked in depth at matched
filters. Specifically, we explored the use of two filters with impulse
responses that are time-reversed versions of the two tones used by an
FSK system. 

We explored the relationship between correlation and the
ouput of the matched filters sampled at the end of a bit period
(i.e., sampled every $f_s{T}$ samples).

To begin, let us turn the operations of matched filtering and sampling (every $f_s T$ samples)
into a function.

### Task: Matched Filter function 

Recall that the matched filter is a filter with an impulse response that is the time-reverse of the signal that it is matched to.

In part 3, you constructed the impulse responses `h0` and `h1` for FSK tones `p0` anf `p1`, respectively. This can be accomplished with the following code:


In [ ]:
# construct the two tones
p0 = fsk_modulator([0], f0, f1, T, fs)
p1 = fsk_modulator([1], f0, f1, T, fs)

# construct the two matched filters
h0 = np.flip(p0)
h1 = np.flip(p1)

The matched filter function you are to write, takes three inputs:
* `sig` - a NumPy array of samples to be filtered
* `hh` - a NumPy array of samples with the impulse response of the matched filter
* `fsT` - an integer $N$ indicating the number of samples per bit period

The function performs the following functions:
* it convolves the input signal `sig` with the impulse response `hh`
* it extracts and returns samples $N-1, 2N-1, 3N-1, ...$, where $N=f_sT$

Name your function `matched_filter`. Hence, your function should start like this:

In [ ]:
def matched_filter(sig, hh, fsT):
    """matched filter
    
    Parameters:
    -----------
    sig - input signal
    hh - impulse response
    fsT - (integer) number of samples per bit period

    Returns:
    --------
    NumPy array of down-sampled matched filter
    """

    y = np.convolve(sig, hh)
    samples = []
    

    N=fsT

    for n in range(N-1, len(sig), fsT):
        samples.append(y[n])
    return np.array(samples)

In [ ]:
# checks
fsT = int(fs*T)

assert np.allclose(matched_filter(p0, h0, fsT), np.sum(p0 ** 2))
assert np.allclose(matched_filter(p1, h1, fsT), np.sum(p1 ** 2))
assert np.allclose(matched_filter(p0, h1, fsT), 0)
assert np.allclose(matched_filter(p1, h0, fsT), 0)

print('Ok')

###  Detector

The remaining task in the receiver is to convert the matched filter
outputs to bits. The receiver's goal is to come up with estimated bits
that are as similar as possible to the transmitted bits. Every bit
that the receiver gets wrong is called a bit error. Thus, the
receiver's goal is to minimize the number of bit errors.

This would be easy if the receiver could observe the transmitted
signal directly. In any realistic communications system, the
transmitted signal becomes weaker as it travels from the transmitter
to the receiver and is corrupted by noise. The connection between
transmitter and receiver is referred to as the *channel*. A
common and important model for a channel is the additive noise channel
that describes that the received signal consists of the sum of transmitted
signal and noise. You will be given a function that simulates
an additive noise channel.

You will learn (in ECE 460) that the best receiver for an additive noise
channel consists of a matched filter frontend followed by a detector.
For our FSK system (and many other systems), the detector is very
simple.

Let us denote the matched filter outputs for the $m$-th
bit period as $y_0[m]$ (for tone `p0`) and  $y_1[m]$ (for tone
`p1`). Then, the optimum detector makes its decision regarding
the $m$-th bit $b_m$ as follows
$$
b_m = \begin{cases}
0 & \text{if $y_0[m] > y_1[m]$}\\
1 & \text{if $y_0[m] < y_1[m]$}
\end{cases}
$$



### Task: Detector Function

Write a function `detector` that converts matched filter outputs into bit decisions.

This function takes as inputs two vectors `y0` and `y1` of matched filter outputs - one from the matched filter for `p0` and the second from the matchd filter for `p1`

The output from this function is a vector of bits (`dtype=np.uint8`) according to the above decision rule.

Thus, your function should start like this:

In [ ]:
def detector(y0, y1):
    """detector for matched filter outputs
    
    Parameters:
    -----------
    y0 - matched filter output for 0 bits
    y1 - matched filter output for 1 bits

    Returns:
    --------
    bit decisions (type np.uint8)
    """
    bits = []

    for n in range(len(y0)):
        if (y0[n] > y1[n]):
            bits.append(0)
        else:
            bits.append(1)

    return np.array(bits, dtype=np.uint8)
    
        
    

In [ ]:
# check
tx_bits = np.array([1, 0, 0, 1])
sig = fsk_modulator(tx_bits, f0, f1, T, fs)

y0 = matched_filter(sig, h0, fsT)
y1 = matched_filter(sig, h1, fsT)

bits = detector(y0, y1)

assert np.all(bits == tx_bits)

print('Ok')

### Task: FSK Demodulator

Write a function that complements the `fsk_modulator` function. Name your function `fsk_demodulator`.

This function takes as inputs:
* `sig` - a NumPy array holding the received signal samples
* `f0` - the first FSK frequency
* `f1` - the second FSK frequency
* `T` - the bit period
* `fs` - the sample rate

Your function must incorporate the functionality developed above, specifically:
* construct the impulse responses `h0` and `h1` for the two matched filters
* pass the signal through both matched filters (function `matched_filter`)
* use the matched filter outputs to form bit decisions (function `detector`)
* return the bit decisions

In [ ]:
def fsk_demodulator(sig, f0, f1, T, fs):
    """FSK demodulator
    
    Parameters:
    -----------
    sig - received signal
    f0 - first FSK frequency
    f1 - second FSK frequency
    T - bit period
    fs - sample rate

    Returns:
    --------
    Numpy vector of bit decsions
    """
    p0=fsk_modulator([0], f0, f1, T, fs)
    p1= fsk_modulator([1], f0, f1, T, fs)

    h0 = np.flip(p0)
    h1= np.flip(p1)
    

    N = int(fs*T)

    bit_zero = matched_filter(sig, h0, N)
    bit_one = matched_filter(sig, h1, N)

    bits = detector(bit_zero, bit_one)

    return bits
    

In [ ]:
# check
tx_bits = np.array([1,0,0,1], dtype=np.uint8)
sig = fsk_modulator(tx_bits, f0, f1, T, fs)
rx_bits = fsk_demodulator(sig, f0, f1, T, fs)
assert np.all(tx_bits == rx_bits)

print('Ok')

## Channel Simulator

The function below simulates the transmission of a signal over an additive white Gaussian noise channel.

It corrupts the input signal by adding random noise to it. 

The strength of the noise is controlled by the parameter `SNR_dB`. SNR stands for *signal-to-noise-ratio*; it measures the strength of the signal relative to the strength of the noise. This quantity is usually expressed in deciBel (dB). 

For example, if the SNR is 0dB then the noise is as strong as the signal. When the SNR is equal to 10dB, then the signal is 10 times stronger than the noise.

The function below can be used to simulate an additive noise channel.

In [ ]:
def additive_noise_channel(sig, SNR_dB, fsT):
    """simulate additive white Gaussian noise
    
    Parameters:
    -----------
    sig - transmitted signal
    SNR_dB - SNR in dB
    fsT - number of samples per symbol period

    Returns:
    --------
    simulated received signal
    """
    N_bits = len(sig) / fsT

    # energy per bit
    Eb = np.sum( sig ** 2) / N_bits

    # linear SNR
    SNR = 10 ** (SNR_dB/10)

    # noise sample variance
    var_noise = Eb/SNR

    # add noise
    return sig + np.sqrt(var_noise) * np.random.randn(len(sig))

## Simulation of the complete system

We now have everything in place to simulate the complete system.

The complete system consists of:
* the transmitter - converts a sequence of bits (or text) into a signal
* the channel - corrupts the transmitted signal by adding noise
* the receiver - seeks to extract the original bits from the noisy signal

Hence, simulation of the complete system can be performed by invoking the functions
* `fsk_modulator` - for turning bits into a signal
* `additive_noise_channel` - to turn a clean signal into a noisy received signal
* `fsk_demodulator` - for extracting bits from the noisy signal


### Experiment: Noisy Signals

In the first experiment, we will assess the impact of noise qualitatively by comparing the transmitted and the received signals for different levels of SNR.

We verify that our system works by simulating an extremely good channel (SNR equal to 100dB); there should be virtually no difference between transmitted and received signals.

In [ ]:
## complete system simulation
# set parameters
tx_bits = np.array([1,0,0,1], dtype=np.uint8)
f0 = 100
f1 = 200
T = 20e-3
fs = 10e3

SNR_dB = 100

# this is the heart of the simulation
sig = fsk_modulator(tx_bits, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# evaluate how it went: any wrong decisions?
print(f'bit errors: {np.sum(rx_bits != tx_bits)} out of {len(tx_bits)}')

# plot the transmitted and received signals
tt = np.arange(len(sig)) / fs
plt.plot(tt * 1e3, rx_sig, label='received')
plt.plot(tt * 1e3, sig, label='transmitted')

plt.grid()
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.legend()

plt.show()

In [ ]:
## complete system simulation
# set parameters
tx_bits = np.array([1,0,0,1], dtype=np.uint8)
f0 = 100
f1 = 200
T = 20e-3
fs = 10e3

SNR_dB = 20

# this is the heart of the simulation
sig = fsk_modulator(tx_bits, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# evaluate how it went: any wrong decisions?
print(f'bit errors: {np.sum(rx_bits != tx_bits)} out of {len(tx_bits)}')

# plot the transmitted and received signals
tt = np.arange(len(sig)) / fs
plt.plot(tt * 1e3, rx_sig, label='received')
plt.plot(tt * 1e3, sig, label='transmitted')

plt.grid()
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.legend()

plt.show()

In [ ]:
## complete system simulation
# set parameters
tx_bits = np.array([1,0,0,1], dtype=np.uint8)
f0 = 100
f1 = 200
T = 20e-3
fs = 10e3

SNR_dB = 10

# this is the heart of the simulation
sig = fsk_modulator(tx_bits, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# evaluate how it went: any wrong decisions?
print(f'bit errors: {np.sum(rx_bits != tx_bits)} out of {len(tx_bits)}')

# plot the transmitted and received signals
tt = np.arange(len(sig)) / fs
plt.plot(tt * 1e3, rx_sig, label='received')
plt.plot(tt * 1e3, sig, label='transmitted')

plt.grid()
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.legend()

plt.show()

In [ ]:
## complete system simulation
# set parameters
tx_bits = np.array([1,0,0,1], dtype=np.uint8)
f0 = 100
f1 = 200
T = 20e-3
fs = 10e3

SNR_dB = 0

# this is the heart of the simulation
sig = fsk_modulator(tx_bits, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# evaluate how it went: any wrong decisions?
print(f'bit errors: {np.sum(rx_bits != tx_bits)} out of {len(tx_bits)}')

# plot the transmitted and received signals
tt = np.arange(len(sig)) / fs
plt.plot(tt * 1e3, rx_sig, label='received')
plt.plot(tt * 1e3, sig, label='transmitted')

plt.grid()
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.legend()

plt.show()

#### Task: Vary the SNR

As expected, the received signal is indistinguishable from the transmitted signal.

Your task is to repeat the above experiment for:
* `SNR_dB=20`
* `SNR_dB=10`
* `SNR_dB=0`

Make a new code cell for each experiment, so that all plots are visible.

For each of these SNR, assess qualitatively how severe the noise is (use statements like "it is still possible to make out the signal"). Also note the number of bit errors as a quantitative measure for the impact of the noise.

### Findings:

The stronger the noise the more distorted the signal we can still kinda of see the signal for 20 but after we start getting bit errors and the original signal is not easily seen so the more stronger the noise is the higher the chances for bit errors to occur.



### Experiment: Sending Text

In the second experiment, we transmit text (the string `'Inf😲rmätion'`). 

Our goal is to identify the lowest SNR level that still permits reliable transmission. Towards that end, we simulate transmission of the string over the additive noise channel. We seek to find an SNR level such that the string is still received without error.

In the process, we also examine the effect of noise in the frequency domain (using the spectrogram function).

In [ ]:
## Experiment: transmit a string
# for a meaningful spectrogram, we need different f0 and f1
f0 = 4e3
f1 = 7e3
fs = 22050

# set SNR
SNR_dB = 30

# simulate transmission of text
tx_str = 'Inf😲rmätion'
sig = fsk_transmitter(tx_str, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# Evaluation
print('Received: ', bits_to_string(rx_bits))

# spectrogram
plt.specgram(rx_sig, Fs=fs)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.colorbar()

plt.show()


You may find it interesting to listen to the received signal to get a sense for how strong the noise is in the received signal.


In [ ]:
# play the received signal
Audio(rx_sig, rate=fs)

### Task: Find the breaking point

At the SNR level for the above figure (30dB), the string is received reliably.

The spectrogram still shows the FSK signal clearly distinguishable from the background level. The increased background level is due to the noise. Note that the noise is spread out, wheras the signal is very concentrated in the spectrogram. This difference is exploited by the matched filter to allow reliable transmission even when the noise is strong.

If you look at the color scale, the difference between the bright yellow signal areas and the average over the (green) noise is approximately 30dB.

Your task is to vary the SNR level and try to find the level (to within 1-2 dB accuracy) where the system starts to break. Copy the above experiment code cell and re-run it with different SNR levels. Since the noise is random, near the breakpoint you will sometimes see an error and sometimes not, even when the SNR is held constant. Look for an SNR value that tends to cause minor errors in the string (e.g., one wrong character). Then increase SNR by 1 dB and check that it tends _not_ to cause errors. 

In [ ]:
## Experiment: transmit a string
# for a meaningful spectrogram, we need different f0 and f1
f0 = 4e3
f1 = 7e3
fs = 22050

# set SNR
SNR_dB = 10

# simulate transmission of text
tx_str = 'Inf😲rmätion'
sig = fsk_transmitter(tx_str, f0, f1, T, fs)
rx_sig = additive_noise_channel(sig, SNR_dB, fs*T)
rx_bits = fsk_demodulator(rx_sig, f0, f1, T, fs)

# Evaluation
print('Received: ', bits_to_string(rx_bits))

# spectrogram
plt.specgram(rx_sig, Fs=fs)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.colorbar()

plt.show()


### Findings
 
State the breaking point that you found and explain how you found it. Include a spectrogram of the received signal at the breaking point.

In repeated trials, the observed breaking point was approximately **10 dB SNR**. At 11 dB the transmitted text was repeatedly recovered correctly, while at 10 dB incorrect characters appeared. The 10 dB spectrogram shows substantial background noise, making the 4 kHz and 7 kHz FSK components difficult to distinguish visually. This is an approximate experimental transition rather than a strict threshold because each transmission uses a different random noise realization.

I kept adjusting the level for SNR_dB until the characters werent being recieved correctly as you can see in the above spectrogram the freq and the background is very hard to distinguish and the recieved text is not the text that was transmitted 

## Summary

This was Part 4 of the Frequency Shift Keying (FSK) project. In this part of the project, we developed code to implement a complete FSK system including: (1) the FSK transmitter, which generates an FSK signal from a text string; (2) a noisy communication channel, which corrupts the transmitted FSK signal; and (3) the FSK receiver, which decodes the bit sequence from the FSK signal using the matched filter and converts the bit sequence into a text string. We also plotted the spectrogram of the received FSK signal.

### Deliverable

Submit a PDF version of this notebook. Make sure that:
* All cells are properly typeset and run.
* All code cells are complete (no `FILL_ME_IN` left).
* All functions are properly documented.
* All plots have proper axis labels and other adornments as appropriate.
* All "Findings" sections are complete.